# Sparse calibration targets

`Target` / `TargetSet` merge observation times into a `SavePlan`, so a
likelihood run never materialises a dense trajectory. Fit an SIR to ~40
scattered observations with `optax`, and show the memory the sparse plan
saves against a daily grid.


In [ ]:
from typing import Any, NamedTuple

import jax
import jax.numpy as jnp
import numpy as np
import optax

from summer4 import (
    Compartments,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
    derived_refs,
)


class Rates(NamedTuple):
    infection: float
    recovery: float


state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)
refs = derived_refs(Rates)
model = FlowModel(pmap)
model.add_flow(TransitionFlow("infection", state["S"], state["I"], refs.infection))
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], refs.recovery))
cm = model.compile()
y0 = PropertyData.wrap(pmap, np.array([999.0, 1.0, 0.0]))
true = Rates(infection=0.35, recovery=0.1)
qty = Compartments(where=state["I"])


## Sparse vs dense footprint

`describe` sizes each output from its own group's `ts`. Pass the same
`params` you would hand to `run` when rates come from a derived struct —
forty observation times are cheaper than an 81-point daily grid.


In [ ]:
rng = np.random.default_rng(0)
obs_t = np.sort(rng.uniform(0.0, 80.0, size=40))
targets = TargetSet(
    targets=(Target(key="I", times=obs_t, values=np.zeros(40), quantity=qty),)
)
sparse = targets.plan(SavePlan())
dense = SavePlan(requests={"I": SaveRequest(qty)})

sparse_desc = cm.describe(sparse, params=true, y0=y0, t0=0.0, dt=1.0, steps=80)
dense_desc = cm.describe(dense, params=true, y0=y0, t0=0.0, dt=1.0, steps=80)
saved_bytes = dense_desc.total_nbytes - sparse_desc.total_nbytes
print(sparse_desc)
print(dense_desc)
print(f"sparse saves {saved_bytes:,} B vs dense daily grid")
assert sparse_desc.total_nbytes < dense_desc.total_nbytes
assert sparse_desc.outputs[0].shape[0] == 40


## Synthetic observations and an optax fit


In [ ]:
truth = cm.run(
    true, y0, t0=0.0, t1=80.0, dt=0.5, save=sparse, solver="dopri5", rtol=1e-7, atol=1e-9
)
obs_vals = np.asarray(truth["I"].at_times(obs_t).values.data).reshape(-1)
targets = TargetSet(
    targets=(Target(key="I", times=obs_t, values=obs_vals, quantity=qty),)
)
plan = targets.plan(SavePlan())


def loss(infection: Any) -> Any:
    params = Rates(infection=infection, recovery=true.recovery)
    res = cm.run(
        params, y0, t0=0.0, t1=80.0, dt=0.5, save=plan, solver="dopri5", rtol=1e-6, atol=1e-8
    )
    return jnp.sum(jnp.asarray(targets.residuals(res)["I"]) ** 2)


opt = optax.adam(0.05)
infection = jnp.asarray(0.15)
opt_state = opt.init(infection)
value_and_grad = jax.jit(jax.value_and_grad(loss))

for _ in range(80):
    _value, grad = value_and_grad(infection)
    updates, opt_state = opt.update(grad, opt_state, infection)
    infection = optax.apply_updates(infection, updates)

recovered = float(infection)
print(f"true infection={true.infection:.3f}, recovered={recovered:.3f}")
assert abs(recovered - true.infection) / true.infection < 0.02
